In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd


In [ ]:
#Also, try the following libraries on the dataset:
#Turicreate
#Surprise

#!pip install turicreate  # Install turicreate
#import turicreate as tc
#!pip install scikit-surprise  # Install Surprise
#from surprise import Dataset, Reader, SVD
#from surprise.model_selection import train_test_split

# I try to use these libraries, bu I could not.



In [ ]:

y!wget https://files.grouplens.org/datasets/movielens/ml-20m.zip
!unzip ml-20m.zip

--2025-03-24 00:43:19--  https://files.grouplens.org/datasets/movielens/ml-20m.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.65.152
Connecting to files.grouplens.org (files.grouplens.org)|128.101.65.152|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 198702078 (189M) [application/zip]
Saving to: ‘ml-20m.zip’

ml-20m.zip          100%[===================>] 189.50M  59.4MB/s    in 3.2s    

2025-03-24 00:43:22 (59.4 MB/s) - ‘ml-20m.zip’ saved [198702078/198702078]

Archive:  ml-20m.zip
replace ml-20m/genome-scores.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ml-20m/genome-scores.csv  
replace ml-20m/genome-tags.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ml-20m/genome-tags.csv  
replace ml-20m/links.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ml-20m/links.csv        
replace ml-20m/movies.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ml-20m/movies.csv       
replace ml-20m/ratings.csv? [y]

 # Read Movies.csv, Ratings.csv and Tags.csv.

In [ ]:
import pandas as pd

# Define column names for each file
ratings_cols = ['userId', 'movieId', 'rating', 'timestamp']
tags_cols = ['userId', 'movieId', 'tag', 'timestamp']
movies_cols = ['movieId', 'title', 'genres']

# Load the dataset with specified column names
ratings = pd.read_csv('ml-20m/ratings.csv', nrows=1000000, names=ratings_cols, header=0) # Limit to 1M rows
movies = pd.read_csv('ml-20m/movies.csv', names=movies_cols, header=0)
tags = pd.read_csv('ml-20m/tags.csv', names=tags_cols, header=0)


In [ ]:
# Display some information about ratings Data
print("\nratings Data :")
print("Ratings data shape:", ratings.shape)
print(ratings.head())


ratings Data :
Ratings data shape: (1000000, 4)
   userId  movieId  rating   timestamp
0       1        2     3.5  1112486027
1       1       29     3.5  1112484676
2       1       32     3.5  1112484819
3       1       47     3.5  1112484727
4       1       50     3.5  1112484580


In [ ]:
# Display some information about movies Data
print("\nmovies Data :")
print("Movies data shape:", movies.shape)
print(movies.head())


movies Data :
Movies data shape: (27278, 3)
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [ ]:
# Display some information about tags Data
print("\ntags Data :")
print("Tags data shape:", tags.shape)
print(tags.head())


tags Data :
Tags data shape: (465564, 4)
   userId  movieId            tag   timestamp
0      18     4141    Mark Waters  1240597180
1      65      208      dark hero  1368150078
2      65      353      dark hero  1368150079
3      65      521  noir thriller  1368149983
4      65      592      dark hero  1368150078


#popularity recommendation

In [ ]:
# popularity recommendation
def popularity_recommendations(top_n=10):
    """
    Recommends movies based on popularity (average rating and number of ratings).

    Args:
        top_n (int): The number of recommendations to generate.

    Returns:
        pandas.DataFrame: A DataFrame containing the top_n recommendations with movie titles, average ratings, and number of ratings.
    """
    # Calculate average rating and number of ratings for each movie
    movie_ratings = ratings.groupby('movieId')['rating'].agg(['mean', 'count'])
    movie_ratings.rename(columns={'mean': 'avg_rating', 'count': 'num_ratings'}, inplace=True)

    # Merge with movie titles
    popular_movies = pd.merge(movie_ratings, movies[['movieId', 'title']], on='movieId')

    # Sort by average rating and number of ratings
    popular_movies = popular_movies.sort_values(['avg_rating', 'num_ratings'], ascending=[False, False])

    # Return the top_n recommendations
    return popular_movies[['title', 'avg_rating', 'num_ratings']].head(top_n)

# Example usage:
popular_recs = popularity_recommendations(10)
print(popular_recs)

                                                   title  avg_rating  \
130                                 Sonic Outlaws (1995)         5.0   
2772           Amor brujo, El (Love Bewitched, A) (1986)         5.0   
3769                                  Boys Life 3 (2000)         5.0   
6260                War and Peace (Jang Aur Aman) (2001)         5.0   
8092                                     Best Boy (1979)         5.0   
10823  Singapore Sling (Singapore sling: O anthropos ...         5.0   
649                                    Tarantella (1995)         5.0   
700                                   War Stories (1995)         5.0   
1308   Forbidden Christ, The (Cristo proibito, Il) (1...         5.0   
1401                               Rhyme & Reason (1997)         5.0   

       num_ratings  
130              3  
2772             3  
3769             2  
6260             2  
8092             2  
10823            2  
649              1  
700              1  
1308             1

# Create content filtering method on metadata obtained from merging movies and tags.
### Metadata should be formed from joining all tag field for each movie_title.



In [ ]:
# content filtering

# Merge movies and tags dataframes
movie_tags = pd.merge(movies, tags, on='movieId', how='left')

# Group by movieId and join all tags for each movie
movie_tags = movie_tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x.astype(str))).reset_index()

# Merge the metadata back into the movies dataframe
movies = pd.merge(movies, movie_tags, on='movieId', how='left')

# Fill NaN values in 'tag' column with empty string
movies['tag'] = movies['tag'].fillna('')

# Create the metadata column by combining title and tags
movies['metadata'] = movies['title'] + ' ' + movies['tag']

# Display the first few rows of the updated movies dataframe
print(movies.head())


   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  \
0  Adventure|Animation|Children|Comedy|Fantasy   
1                   Adventure|Children|Fantasy   
2                               Comedy|Romance   
3                         Comedy|Drama|Romance   
4                                       Comedy   

                                                 tag  \
0  Watched computer animation Disney animated fea...   
1  time travel adapted from:book board game child...   
2  old people that is actually funny sequel fever...   
3  chick flick revenge characters chick flick cha...   
4  Diane Keaton family sequel Steve Martin weddin...   

                                            metadata  
0  Toy

# Build a Tfidf Vectorizer model and TruncatedSVD for Content filter - Latent matrix 1 on this data



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Create a TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words='english')

# Fit and transform the metadata to create the TF-IDF matrix
tfidf_matrix = tfidf.fit_transform(movies['metadata'])

# Create a TruncatedSVD model
svd = TruncatedSVD(n_components=200)  # You can adjust the number of components

# Fit and transform the TF-IDF matrix to create the latent matrix
latent_matrix_1 = svd.fit_transform(tfidf_matrix)

# Print the shape of the latent matrix
print("Latent Matrix 1 shape:", latent_matrix_1.shape)

Latent Matrix 1 shape: (27278, 200)


# Create a Collab filter on User Movie matrix (formed from pivot table on ratings data

In [ ]:
# Collab filtering

from scipy.sparse import csr_matrix

# Create a user-movie matrix using pivot_table
user_movie_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)

# Convert the matrix to a sparse matrix for efficiency
user_movie_matrix_sparse = csr_matrix(user_movie_matrix.values)


# Create a Latent matrix 2 on this data

In [ ]:
from sklearn.decomposition import TruncatedSVD

# Create a TruncatedSVD model
svd = TruncatedSVD(n_components=200)  # You can adjust the number of components

# Fit and transform the user-movie matrix to create Latent Matrix 2
latent_matrix_2 = svd.fit_transform(user_movie_matrix_sparse)

# Print the shape of Latent Matrix 2
print("Latent Matrix 2 shape:", latent_matrix_2.shape)

Latent Matrix 2 shape: (6743, 200)


# Code hybrid model

In [ ]:
# hybrid filtering

from sklearn.metrics.pairwise import cosine_similarity

def hybrid_recommendations(movie_title, top_n=10):
    """
    Generates hybrid recommendations by combining content and collaborative filtering.

    Args:
        movie_title (str): The title of the movie to get recommendations for.
        top_n (int): The number of recommendations to generate.

    Returns:
        pandas.DataFrame: A DataFrame containing the top_n recommendations with movie titles and scores.
    """
    # Get the index of the movie in the movies DataFrame
    movie_index = movies[movies['title'] == movie_title].index[0]

    # Get the content-based similarity scores
    content_scores = cosine_similarity(latent_matrix_1[movie_index].reshape(1, -1), latent_matrix_1)

    # Get the movieId of the input movie
    movie_id = movies.loc[movie_index, 'movieId']

    # Check if the movieId is in the columns of the user_movie_matrix
    if movie_id in user_movie_matrix.columns:
        # Get the column index corresponding to the movieId in user_movie_matrix
        movie_col_index = user_movie_matrix.columns.get_loc(movie_id)

        # Get the collaborative filtering similarity scores for users who rated the movie
        # Filter latent_matrix_2 to include only users who rated the movie
        users_rated_movie = user_movie_matrix[user_movie_matrix[movie_id] > 0].index
        # Fix: Subtracting 1 to convert to 0-based indexing
        filtered_latent_matrix_2 = latent_matrix_2[users_rated_movie - 1]

        # Calculate collaborative scores using the filtered matrix
        collab_scores = cosine_similarity(filtered_latent_matrix_2[movie_col_index].reshape(1, -1), filtered_latent_matrix_2)

        # Resize collab_scores to match content_scores
        collab_scores = np.resize(collab_scores, content_scores.shape)  # Resize collab_scores

        # Combine the scores (adjust weights as needed)
        hybrid_scores = (0.5 * content_scores + 0.5 * collab_scores).flatten()

        # Get the indices of the top_n most similar movies (excluding the input movie)
        similar_movie_indices = hybrid_scores.argsort()[:-top_n-2:-1][1:]  # Exclude input movie and take top_n

        # Return the top_n recommendations as a DataFrame
        recommendations = movies.iloc[similar_movie_indices][['title']]
        recommendations['score'] = hybrid_scores[similar_movie_indices]
        return recommendations
    else:
        print(f"Movie '{movie_title}' not found in user-item interactions. Returning content-based recommendations.")

        # If movie not in user-item interactions, fall back to content-based recommendations
        similar_movie_indices = content_scores.argsort()[:-top_n-2:-1][1:]
        recommendations = movies.iloc[similar_movie_indices][['title']]
        recommendations['score'] = content_scores[0, similar_movie_indices]
        return recommendations

# Example usage:
recommendations = hybrid_recommendations("Toy Story (1995)")
print(recommendations)

                                   title     score
21978          Blue Umbrella, The (2013)  0.848232
10565              Chicken Little (2005)  0.638094
6271                 Finding Nemo (2003)  0.636894
25463  Toy Story That Time Forgot (2014)  0.630903
2270                Bug's Life, A (1998)  0.619735
13337     Tale of Despereaux, The (2008)  0.596909
11423                Flushed Away (2006)  0.587690
15526               Despicable Me (2010)  0.580508
14808    Dante's Inferno Animated (2010)  0.577972
4884   Bill & Ted's Bogus Journey (1991)  0.559009
